In [1]:
# IMPORTING DEPENDENCIES

import gc
import sqlite3
import warnings

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, brier_score_loss,
    classification_report, confusion_matrix, log_loss,
)
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")


In [2]:
# HELPER FUNCTION
# TO DOWNCAST NUMERIC COLUMNS TO SAVE RAM
def downcast(df: pd.DataFrame) -> pd.DataFrame:
    for col in df.select_dtypes("float64").columns:
        df[col] = df[col].astype("float32")
    for col in df.select_dtypes("int64").columns:
        df[col] = pd.to_numeric(df[col], downcast="integer")
    return df

In [3]:
# LOADING THE DATA

# LOADING ONLY COLUMNS WE NEED TO SAVE RAM
MATCH_COLS = [
    "date", "year", "month", "home_team", "away_team",
    "home_score", "away_score", "result",
    "tournament", "tournament_tier", "neutral",
]
ELO_COLS = [
    "date", "home_team", "away_team",
    "home_elo_pre", "away_elo_pre", "elo_diff", "exp_home_win_prob",
]
FUTURE_COLS = ["date", "home_team", "away_team", "tournament", "neutral"]

# LOADING DATASET
matches = pd.read_csv(
    "clean_matches.csv",
    usecols=MATCH_COLS, parse_dates=["date"],
)
elo = pd.read_csv(
    "elo_matches.csv",
    usecols=ELO_COLS, parse_dates=["date"],
)
ratings = pd.read_csv(
    "elo_final_ratings.csv",
    usecols=["team", "elo_rating"],
)
future = pd.read_csv(
    "future_matches.csv",
    usecols=FUTURE_COLS, parse_dates=["date"],
)


In [4]:
# DOWNCASTING FLOATS/INTS
downcast(elo)
downcast(matches)

,date,year,month,home_team,away_team,home_score,away_score,result,tournament,tournament_tier,neutral
0,1872-11-30,1872,11,Scotland,England,0,0,draw,Friendly,4,False
1,1873-03-08,1873,3,England,Scotland,4,2,home_win,Friendly,4,False
2,1874-03-07,1874,3,Scotland,England,2,1,home_win,Friendly,4,False
3,1875-03-06,1875,3,England,Scotland,2,2,draw,Friendly,4,False
4,1876-03-04,1876,3,Scotland,England,3,0,home_win,Friendly,4,False
...,...,...,...,...,...,...,...,...,...,...,...
49210,2026-03-31,2026,3,Kosovo,Turkey,0,1,away_win,FIFA World Cup qualification,2,False
49211,2026-03-31,2026,3,Czech Republic,Denmark,2,2,draw,FIFA World Cup qualification,2,False
49212,2026-03-31,2026,3,Cameroon,China PR,2,0,home_win,FIFA Series,3,True
49213,2026-03-31,2026,3,Australia,Curaçao,5,1,home_win,FIFA Series,3,False


In [5]:
# USING CATEGORY dtype FOR REPEATED STRING COLUMNS
for col in ["home_team", "away_team"]:
    matches[col] = matches[col].astype("category")
    elo[col]     = elo[col].astype("category")

In [6]:
# INITIAL DATASET INSPECTION
print(f"matches : {len(matches):,}")
print(f"elo     : {len(elo):,}")
print(f"ratings : {len(ratings)}")
print(f"future  : {len(future)}")


matches : 49,215
elo     : 49,215
ratings : 333
future  : 72


## FEATURE ENGINEERING

In [7]:
# MERGING ELO INTO MATCHES DATAFRAME

df = matches.merge(elo, on = ["date", "home_team", "away_team"], how = "left")
df.sort_values("date", inplace = True)
df.reset_index(drop = True, inplace = True)


In [8]:
# DROPING ELO NOW
del elo
gc.collect()

612

In [9]:
df.dropna(subset = ["home_elo_pre", "away_elo_pre"], inplace = True)
df.reset_index(drop = True, inplace = True)

print(f"After ELO filter: {len(df):,} rows")

After ELO filter: 49,217 rows


In [10]:
# CONVERTING TEAM COLUMNS BACK FROM CATEGORY
df["home_team"] = df["home_team"].astype(str)
df["away_team"] = df["away_team"].astype(str)


In [11]:
# BUILDING ROLLING FORM
print("Building rolling form + goal features (single pass)...")

home_view = df[["date", "home_team", "result", "home_score", "away_score"]].copy()
home_view.columns = ["date", "team", "raw_result", "scored", "conceded"]
home_view["win"]  = (home_view["raw_result"] == "home_win").astype("int8")
home_view["draw"] = (home_view["raw_result"] == "draw").astype("int8")
home_view["loss"] = (home_view["raw_result"] == "away_win").astype("int8")
home_view["side"] = "home"

away_view = df[["date", "away_team", "result", "away_score", "home_score"]].copy()
away_view.columns = ["date", "team", "raw_result", "scored", "conceded"]
away_view["win"]  = (away_view["raw_result"] == "away_win").astype("int8")
away_view["draw"] = (away_view["raw_result"] == "draw").astype("int8")
away_view["loss"] = (away_view["raw_result"] == "home_win").astype("int8")
away_view["side"] = "away"

combined = pd.concat([home_view, away_view], ignore_index=True)
combined.sort_values("date", inplace=True)
del home_view, away_view
gc.collect()

# ROLLING COMPUTATION
for metric in ["win", "draw", "loss"]:
    for w in (5, 10):
        combined[f"{metric}_rate_{w}"] = (
            combined.groupby("team")[metric]
            .transform(lambda x: x.shift(1).rolling(w, min_periods = 1).mean())
            .astype("float32")
        )

for col in ["scored", "conceded"]:
    combined[f"avg_{col}_10"] = (
        combined.groupby("team")[col]
        .transform(lambda x: x.shift(1).rolling(10, min_periods = 3).mean())
        .astype("float32")
    )

# SPLIT INTO HOME/AWAY LOOKUP TABLE AND MERGE ONCE EACH
roll_cols_base = [
    "win_rate_5", "draw_rate_5", "loss_rate_5",
    "win_rate_10", "draw_rate_10", "loss_rate_10",
    "avg_scored_10", "avg_conceded_10",
]

for side in ("home", "away"):
    sub = combined[combined["side"] == side][
        ["date", "team"] + roll_cols_base
    ].rename(columns={
        "team": f"{side}_team",
        **{c: f"{side}_{c}" for c in roll_cols_base},
    })
    df = df.merge(sub, on=["date", f"{side}_team"], how = "left")

del combined, sub
gc.collect()
print("Rolling form + goal features built")



Building rolling form + goal features (single pass)...
Rolling form + goal features built


In [12]:
# HEAD-TO-HEAD
print("Building H2H features...")

df["h2h_pair"] = [
    "_vs_".join(sorted([h, a]))
    for h, a in zip(df["home_team"], df["away_team"])
]

pair_df = df[["date", "h2h_pair", "home_team", "away_team", "result"]].copy()
pair_df.sort_values("date", inplace = True)
pair_df["pair_home_win"] = (pair_df["result"] == "home_win").astype("int8")
pair_df["pair_match"]    = np.int8(1)

pair_df["cum_home_wins"] = (
    pair_df.groupby("h2h_pair")["pair_home_win"]
    .transform(lambda x: x.shift(1).cumsum())
)

pair_df["cum_total"] = (
    pair_df.groupby("h2h_pair")["pair_match"]
    .transform(lambda x: x.shift(1).cumsum())
)

pair_df["h2h_home_win_rate"] = np.where(
    pair_df["cum_total"] >= 3,
    (pair_df["cum_home_wins"] / pair_df["cum_total"]).astype("float32"),
    np.nan,
)

pair_df["h2h_total_matches"] = pair_df["cum_total"].fillna(0).astype("float32")

df = df.merge(
    pair_df[["date", "home_team", "away_team",
             "h2h_home_win_rate", "h2h_total_matches"]],
    on = ["date", "home_team", "away_team"], how = "left",
)

df.drop(columns = ["h2h_pair"], inplace = True)
del pair_df
gc.collect()

print("H2H built")

Building H2H features...
H2H built


In [13]:
# REST DAYS FEATURE

print("Building days rest feature...")

all_app = pd.concat([
    df[["date", "home_team"]].rename(columns={"home_team": "team"}),
    df[["date", "away_team"]].rename(columns={"away_team": "team"}),
], ignore_index=True).sort_values("date")

all_app["prev_date"] = all_app.groupby("team")["date"].shift(1)
all_app["days_since_last"] = (all_app["date"] - all_app["prev_date"]).dt.days.astype("float32")

df = df.merge(
    all_app.rename(columns={"team": "home_team", "days_since_last": "days_rest_home"})
          [["date", "home_team", "days_rest_home"]]
          .drop_duplicates(["date", "home_team"]),
    on = ["date", "home_team"], how = "left",
)

df = df.merge(
    all_app.rename(columns={"team": "away_team", "days_since_last": "days_rest_away"})
          [["date", "away_team", "days_rest_away"]]
          .drop_duplicates(["date", "away_team"]),
    on = ["date", "away_team"], how = "left",
)

df["rest_advantage"] = (
    df["days_rest_home"].fillna(30) - df["days_rest_away"].fillna(30)
).astype("float32")


del all_app
gc.collect()

print("Days rest built")


Building days rest feature...
Days rest built


In [14]:
# ENCODING TARGET AND CONTEXT

result_map = {"away_win": 0, "draw": 1, "home_win": 2}

df["target"] = df["result"].map(result_map).astype("int8")
df["neutral"] = df["neutral"].astype("int8")
df["tournament_tier"] = df["tournament_tier"].astype("int8")


In [15]:
# MAKING LIST OF FINAL FEATURE SET

FEATURES = [
    "home_elo_pre", "away_elo_pre", "elo_diff", "exp_home_win_prob",
    "home_win_rate_5", "home_draw_rate_5",
    "away_win_rate_5", "away_draw_rate_5",
    "home_win_rate_10", "away_win_rate_10",
    "home_avg_scored_10", "home_avg_conceded_10",
    "away_avg_scored_10", "away_avg_conceded_10",
    "h2h_home_win_rate", "h2h_total_matches",
    "tournament_tier", "neutral", "rest_advantage", "month",
]

df_ml = df.dropna(subset=["home_win_rate_5", "away_win_rate_5",
                           "home_avg_scored_10"]).copy()

df_ml = df_ml.dropna(subset = FEATURES).reset_index(drop = True)
df_ml["h2h_home_win_rate"].fillna(0.4, inplace = True)
df_ml["h2h_total_matches"].fillna(0,   inplace = True)

# DROPING THE LARGE RAW df
# KEEPING ONLY df_ml
del df
gc.collect()

X = df_ml[FEATURES]
y = df_ml["target"]

print(f"\nML dataset: {len(df_ml):,} rows × {len(FEATURES)} features")
print(f"  Target distribution:")
print(y.value_counts().sort_index().rename({0: "Away win", 1: "Draw", 2: "Home win"}))
df_ml.to_csv("ml_dataset.csv", index=False)




ML dataset: 36,452 rows × 20 features
  Target distribution:
target
Away win    10889
Draw         7852
Home win    17711
Name: count, dtype: int64


## LOGISTIC REGRESSION BASELINE

In [16]:
print("Baseline — Logistic Regression")

split_date = pd.Timestamp("2015-01-01")
train_mask = df_ml["date"] < split_date
test_mask  = df_ml["date"] >= split_date

X_train, X_test = X[train_mask], X[test_mask]
y_train, y_test = y[train_mask], y[test_mask]

print(f"Train: {len(X_train):,} matches (pre-2015)")
print(f"Test : {len(X_test):,} matches (2015+)")

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

lr = LogisticRegression(
    max_iter = 1000, C = 1.0, solver = "lbfgs", random_state = 42,
)

lr.fit(X_train_sc, y_train)

y_pred_lr  = lr.predict(X_test_sc)
y_proba_lr = lr.predict_proba(X_test_sc)

acc_lr = accuracy_score(y_test, y_pred_lr)
ll_lr  = log_loss(y_test, y_proba_lr)

bs_lr  = np.mean([
    brier_score_loss((y_test == c).astype(int), y_proba_lr[:, c])
    for c in range(3)
])


print(f"Accuracy: {acc_lr:.4f}  Log-loss: {ll_lr:.4f}  Brier: {bs_lr:.4f}")
print(classification_report(y_test, y_pred_lr,
                             target_names=["Away Win", "Draw", "Home Win"]))



Baseline — Logistic Regression
Train: 28,750 matches (pre-2015)
Test : 7,702 matches (2015+)
Accuracy: 0.5780  Log-loss: 0.9012  Brier: 0.1775
              precision    recall  f1-score   support

    Away Win       0.58      0.53      0.55      2189
        Draw       0.31      0.06      0.10      1876
    Home Win       0.60      0.88      0.71      3637

    accuracy                           0.58      7702
   macro avg       0.50      0.49      0.46      7702
weighted avg       0.52      0.58      0.52      7702



## XGBOOST MODEL

In [19]:
print("XGBoost model")

val_date    = pd.Timestamp("2012-01-01")
val_mask    = (df_ml["date"] >= val_date) & (df_ml["date"] < split_date)
train_mask2 = df_ml["date"] < val_date

X_tr2 = X[train_mask2]
y_tr2 = y[train_mask2]
X_val = X[val_mask]
y_val = y[val_mask]

xgb_model = xgb.XGBClassifier(
    n_estimators = 500,
    max_depth = 4,
    learning_rate = 0.05,
    subsample = 0.8,
    colsample_bytree = 0.8,
    min_child_weight = 10,
    gamma = 1,
    reg_alpha = 0.1,
    reg_lambda = 1.0,
    objective = "multi:softprob",
    num_class = 3,
    eval_metric = "mlogloss",
    early_stopping_rounds = 30,
    random_state = 42,
    n_jobs = -1,
    tree_method = "hist",
    device = "cpu",
)

xgb_model.fit(
    X_tr2, y_tr2,
    eval_set = [(X_val, y_val)],
    verbose = 100,
)

y_pred_xgb  = xgb_model.predict(X_test)
y_proba_xgb = xgb_model.predict_proba(X_test)

acc_xgb = accuracy_score(y_test, y_pred_xgb)
ll_xgb  = log_loss(y_test, y_proba_xgb)
bs_xgb  = np.mean([
    brier_score_loss((y_test == c).astype(int), y_proba_xgb[:, c])
    for c in range(3)
])


print(f"Accuracy: {acc_xgb:.4f}  Log-loss: {ll_xgb:.4f}  Brier: {bs_xgb:.4f}")
print(classification_report(y_test, y_pred_xgb,
                             target_names=["Away Win", "Draw", "Home Win"]))

cm = confusion_matrix(y_test, y_pred_xgb)

print("  Confusion Matrix (rows=actual, cols=predicted):")
print(f"              Away  Draw  Home")
for label, row in zip(["Away Win ", "Draw     ", "Home Win "], cm):
    print(f"  {label}: {row}")



XGBoost model
[0]	validation_0-mlogloss:1.04622
[100]	validation_0-mlogloss:0.93385
[200]	validation_0-mlogloss:0.93306
[300]	validation_0-mlogloss:0.93303
[305]	validation_0-mlogloss:0.93307
Accuracy: 0.5824  Log-loss: 0.9003  Brier: 0.1772
              precision    recall  f1-score   support

    Away Win       0.57      0.56      0.57      2189
        Draw       0.37      0.04      0.07      1876
    Home Win       0.59      0.88      0.71      3637

    accuracy                           0.58      7702
   macro avg       0.51      0.49      0.45      7702
weighted avg       0.53      0.58      0.51      7702

  Confusion Matrix (rows=actual, cols=predicted):
              Away  Draw  Home
  Away Win : [1222   63  904]
  Draw     : [ 527   67 1282]
  Home Win : [ 387   53 3197]


## EVALUATION

In [21]:
print("Model evaluation")


rng = np.random.default_rng(42)
y_naive_home    = np.full(len(y_test), 2)
y_naive_random  = rng.choice([0, 1, 2], len(y_test))
y_proba_uniform = np.full((len(y_test), 3), 1 / 3)

acc_naive_home   = accuracy_score(y_test, y_naive_home)
acc_naive_random = accuracy_score(y_test, y_naive_random)
ll_naive_uniform = log_loss(y_test, y_proba_uniform)

print(f"{'Model':<25} {'Accuracy':>10} {'Log-loss':>10} {'Brier':>10}")
print(f"{'-'*57}")
print(f"{'Naive (always home)':<25} {acc_naive_home:>10.4f} {'N/A':>10} {'N/A':>10}")
print(f"{'Logistic Regression':<25} {acc_lr:>10.4f} {ll_lr:>10.4f} {bs_lr:>10.4f}")
print(f"{'XGBoost':<25} {acc_xgb:>10.4f} {ll_xgb:>10.4f} {bs_xgb:>10.4f}")

eval_df = pd.DataFrame([
    {"model": "Naive (always home)", "accuracy": acc_naive_home, "log_loss": None, "brier": None},
    {"model": "Logistic Regression", "accuracy": acc_lr, "log_loss": ll_lr, "brier": bs_lr},
    {"model": "XGBoost",             "accuracy": acc_xgb, "log_loss": ll_xgb, "brier": bs_xgb},
])

eval_df.to_csv("ml_model_evaluation.csv", index=False)


print("\n\nYear-by-year XGBoost accuracy:")
test_df = df_ml[test_mask].copy()
test_df["predicted"] = y_pred_xgb
test_df["correct"] = (test_df["predicted"] == test_df["target"]).astype("int8")
for year, acc in test_df.groupby("year")["correct"].mean().items():
    print(f"{int(year)}: {'█' * int(acc * 30)} {acc:.3f}")
del test_df


Model evaluation
Model                       Accuracy   Log-loss      Brier
---------------------------------------------------------
Naive (always home)           0.4722        N/A        N/A
Logistic Regression           0.5780     0.9012     0.1775
XGBoost                       0.5824     0.9003     0.1772


Year-by-year XGBoost accuracy:
2015: █████████████████ 0.573
2016: ██████████████████ 0.600
2017: ████████████████ 0.551
2018: ███████████████ 0.521
2019: █████████████████ 0.597
2020: ████████████████ 0.535
2021: ██████████████████ 0.627
2022: █████████████████ 0.582
2023: ██████████████████ 0.603
2024: █████████████████ 0.572
2025: ██████████████████ 0.603
2026: ██████████████ 0.495


## FEATURE IMPORTANCE

In [22]:
print("Feature importance")


importance_df = pd.DataFrame({
    "feature": FEATURES,
    "importance": xgb_model.feature_importances_,
}).sort_values("importance", ascending = False).reset_index(drop = True)

importance_df["rank"] = importance_df.index + 1
importance_df["importance_pct"] = (
    importance_df["importance"] / importance_df["importance"].sum() * 100
).round(2)

print(f"{'Rank':<5} {'Feature':<28} {'% of total':>12}")
print(f"{'-'*47}")

for _, row in importance_df.iterrows():
    bar = "▓" * int(row["importance_pct"] * 2)
    print(f"{int(row['rank']):<5} {row['feature']:<28} {row['importance_pct']:>10.1f}%  {bar}")

importance_df.to_csv("ml_feature_importance.csv", index=False)


Feature importance
Rank  Feature                        % of total
-----------------------------------------------
1     exp_home_win_prob                  23.8%  ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
2     elo_diff                           12.8%  ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
3     month                               7.6%  ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
4     home_avg_scored_10                  7.5%  ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
5     home_avg_conceded_10                6.4%  ▓▓▓▓▓▓▓▓▓▓▓▓
6     away_avg_conceded_10                4.4%  ▓▓▓▓▓▓▓▓
7     neutral                             3.3%  ▓▓▓▓▓▓
8     home_elo_pre                        3.1%  ▓▓▓▓▓▓
9     home_win_rate_10                    3.1%  ▓▓▓▓▓▓
10    h2h_total_matches                   3.1%  ▓▓▓▓▓▓
11    tournament_tier                     3.1%  ▓▓▓▓▓▓
12    away_elo_pre                        2.8%  ▓▓▓▓▓
13    away_avg_scored_10                  2.6%  ▓▓▓▓▓
14    away_draw_rate_5                    2.5%  ▓▓▓▓
15    away_win_rate_5            

## PREDICT 2026 WORLD CUP FIXTURES

In [24]:
print("Predicting 2026 World Cup fixtures")

latest_elo = ratings.set_index("team")["elo_rating"].to_dict()

def build_team_stats(matches_df: pd.DataFrame) -> dict:
    home_m = matches_df[["home_team", "result", "home_score", "away_score"]].rename(
        columns = {"home_team": "team", "home_score": "scored", "away_score": "conceded"}
    ).assign(
        win  = lambda x: (x["result"] == "home_win").astype("int8"),
        draw = lambda x: (x["result"] == "draw").astype("int8"),
    )

    away_m = matches_df[["away_team", "result", "away_score", "home_score"]].rename(
        columns={"away_team": "team", "away_score": "scored", "home_score": "conceded"}
    ).assign(
        win  = lambda x: (x["result"] == "away_win").astype("int8"),
        draw = lambda x: (x["result"] == "draw").astype("int8"),
    )

    all_m = pd.concat([home_m, away_m], ignore_index = True)

    stats = {}
    for team, grp in all_m.groupby("team"):
        last10 = grp.tail(10)
        last5 = grp.tail(5)
        n10, n5 = len(last10), len(last5)
        stats[team] = {
            "wr5":  last5["win"].sum()  / n5  if n5  else 0.33,
            "dr5":  last5["draw"].sum() / n5  if n5  else 0.33,
            "wr10": last10["win"].sum() / n10 if n10 else 0.33,
            "sc10": last10["scored"].mean()   if n10 else 1.0,
            "co10": last10["conceded"].mean() if n10 else 1.0,
        }

    return stats

team_stats = build_team_stats(matches)
DEFAULT_STATS = {"wr5": 0.33,
                 "dr5": 0.33,
                 "wr10": 0.33,
                 "sc10": 1.0,
                 "co10": 1.0}

wc2026 = future[future["tournament"] == "FIFA World Cup"].copy()

print(f"Found {len(wc2026)} 2026 World Cup fixtures")

if len(wc2026) == 0:
    print("WARNING: No 2026 WC fixtures found — using all future matches")
    wc2026 = future.head(20).copy()

# BUILDING FEATURE MATRIX FOR ALL FIXTURES AT ONCE
rows = []
for _, row in wc2026.iterrows():
    home, away = row["home_team"], row["away_team"]
    hs = team_stats.get(home, DEFAULT_STATS)
    as_ = team_stats.get(away, DEFAULT_STATS)
    h_elo = latest_elo.get(home, 1000.0)
    a_elo = latest_elo.get(away, 1000.0)

    rows.append([
        h_elo, a_elo, h_elo - a_elo,
        1 / (1 + 10 ** ((a_elo - h_elo) / 400)),
        hs["wr5"],  hs["dr5"],
        as_["wr5"], as_["dr5"],
        hs["wr10"], as_["wr10"],
        hs["sc10"], hs["co10"],
        as_["sc10"], as_["co10"],
        0.4, 5.0,
        1, int(row.get("neutral", True)),
        0,
        pd.to_datetime(row["date"]).month,
    ])

feat_matrix = pd.DataFrame(rows, columns = FEATURES)
all_proba = xgb_model.predict_proba(feat_matrix)

predictions = []
for i, (_, row) in enumerate(wc2026.iterrows()):
    proba = all_proba[i]
    predictions.append({
        "date":             row["date"],
        "home_team":        row["home_team"],
        "away_team":        row["away_team"],
        "home_elo":         round(latest_elo.get(row["home_team"], 1000), 0),
        "away_elo":         round(latest_elo.get(row["away_team"], 1000), 0),
        "prob_home_win":    round(proba[2] * 100, 1),
        "prob_draw":        round(proba[1] * 100, 1),
        "prob_away_win":    round(proba[0] * 100, 1),
        "predicted_result": ["Away Win", "Draw", "Home Win"][np.argmax(proba)],
        "favourite":        row["home_team"] if latest_elo.get(row["home_team"], 1000) >
                                               latest_elo.get(row["away_team"], 1000) else row["away_team"],
    })

pred_df = pd.DataFrame(predictions)

if len(pred_df) > 0:
    print(f"\n  {'Date':<12} {'Home':<20} {'Away':<20} {'H%':>5} {'D%':>5} {'A%':>5} {'Predicted'}")
    print(f"  {'-'*85}")
    for _, r in pred_df.iterrows():
        print(f"  {str(r['date'])[:10]:<12} {r['home_team']:<20} {r['away_team']:<20} "
              f"{r['prob_home_win']:>5.1f} {r['prob_draw']:>5.1f} {r['prob_away_win']:>5.1f} "
              f"{r['predicted_result']}")
    pred_df.to_csv("ml_predictions_2026.csv", index = False)


Predicting 2026 World Cup fixtures
Found 72 2026 World Cup fixtures

  Date         Home                 Away                    H%    D%    A% Predicted
  -------------------------------------------------------------------------------------
  2026-06-11   Mexico               South Africa          64.7  23.2  12.2 Home Win
  2026-06-11   South Korea          Czech Republic        47.1  24.5  28.4 Home Win
  2026-06-12   Canada               Bosnia and Herzegovina  70.4  17.8  11.8 Home Win
  2026-06-12   United States        Paraguay              34.0  28.1  38.0 Away Win
  2026-06-13   Qatar                Switzerland           10.6  13.6  75.8 Away Win
  2026-06-13   Brazil               Morocco               46.1  29.1  24.7 Home Win
  2026-06-13   Haiti                Scotland              28.7  24.3  47.0 Away Win
  2026-06-13   Australia            Turkey                34.7  29.1  36.2 Away Win
  2026-06-14   Germany              Curaçao               74.0  21.5   4.6 Home Win


## EXPORTING

In [25]:
print("Saving to SQLite database")


conn = sqlite3.connect("football_analytics.db")
df_ml.to_sql("ml_dataset", conn, if_exists="replace", index = False)
eval_df.to_sql("ml_evaluation", conn, if_exists = "replace", index = False)
importance_df.to_sql("ml_feature_importance", conn, if_exists = "replace", index = False)

if len(pred_df) > 0:
    pred_df.to_sql("ml_predictions_2026", conn, if_exists = "replace", index = False)

conn.close()

print("All tables written → football_analytics.db")



Saving to SQLite database
All tables written → football_analytics.db


## VISUALIZATION

In [26]:
!pip install kaleido==0.2.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 MB 9.1 MB/s eta 0:00:00


In [27]:
import plotly.graph_objects as go
import plotly.io as pio

print("\nGenerating predictions chart...")

plot_df = pred_df.sort_values("prob_home_win", ascending=True).copy()

labels     = plot_df["home_team"] + " vs " + plot_df["away_team"]
home_probs = plot_df["prob_home_win"]
draw_probs = plot_df["prob_draw"]
away_probs = plot_df["prob_away_win"]

customdata = list(zip(
    plot_df["home_team"], plot_df["away_team"],
    home_probs, draw_probs, away_probs,
    plot_df["predicted_result"], plot_df["favourite"],
    plot_df["date"].astype(str).str[:10],
    plot_df["home_elo"], plot_df["away_elo"],
))

hover_tmpl = (
    "<b>%{customdata[0]} vs %{customdata[1]}</b><br>"
    "Date: %{customdata[7]}<br>"
    "Home ELO: %{customdata[8]:.0f} | Away ELO: %{customdata[9]:.0f}<br>"
    "──────────────────<br>"
    "🏠 Home Win: <b>%{customdata[2]:.1f}%</b><br>"
    "🤝 Draw:     <b>%{customdata[3]:.1f}%</b><br>"
    "✈️ Away Win: <b>%{customdata[4]:.1f}%</b><br>"
    "Predicted: <b>%{customdata[5]}</b><br>"
    "Favourite: <b>%{customdata[6]}</b>"
    "<extra></extra>"
)

fig = go.Figure()
for name, x_vals, color in [
    ("Home Win", home_probs, "#3b82f6"),
    ("Draw",     draw_probs, "#64748b"),
    ("Away Win", away_probs, "#f97316"),
]:
    fig.add_trace(go.Bar(
        name = name, y = labels, x = x_vals, orientation = "h",
        marker_color = color,
        customdata = customdata, hovertemplate = hover_tmpl,
        text = [f"{v:.1f}%" for v in x_vals],
        textposition = "inside", insidetextanchor = "middle",
        textfont = dict(size = 11, color = "white"),
    ))

fig.update_layout(
    barmode = "stack",
    title = dict(
        text = "⚽ 2026 FIFA World Cup — Match Outcome Probabilities",
        font = dict(size = 20, color = "#f1f5f9"), x = 0.01,
    ),
    xaxis = dict(title = "Probability (%)", ticksuffix = "%", range = [0, 100],
               gridcolor = "#1e293b", color = "#94a3b8"),
    yaxis = dict(tickfont = dict(size = 11), color = "#cbd5e1", automargin = True),
    legend = dict(orientation = "h", yanchor = "bottom", y = 1.02,
                xanchor = "right", x = 1, font = dict(color = "#e2e8f0")),
    plot_bgcolor = "#0f172a", paper_bgcolor = "#0f172a",
    font = dict(family = "'Segoe UI', system-ui, sans-serif", color = "#e2e8f0"),
    height = max(500, len(plot_df) * 38 + 120),
    margin = dict(l = 20, r = 30, t = 70, b = 50),
    hoverlabel = dict(bgcolor = "#1e293b", bordercolor = "#334155",
                    font_size = 13, font_color = "#f1f5f9"),
)


fig.show()

pio.write_image(fig, "ml_predictions_chart.png", format="png",
                width=1400, height=max(600, len(plot_df) * 38 + 120), scale=2)


Generating predictions chart...
